In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\rt01284\GITHUB\Evolve-MDS-2025-Oct-IAGen\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
# Modelo que quieres usar
MODEL = "gemini-2.5-flash-lite"

In [ ]:
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7, google_api_key=API_KEY)

In [ ]:
pregunta = "¿En qué año llegó el ser humano a la Luna por primera vez?"
print("Pregunta: ", pregunta)

Pregunta:  ¿En qué año llegó el ser humano a la Luna por primera vez?


## Invocation

Invoke

In [ ]:
respueta = llm.invoke(pregunta)
print("Respuesta del modelo: ", respueta.content)

Respuesta del modelo:  El ser humano llegó a la Luna por primera vez en **1969**.


Stream

In [ ]:
for chunk in llm.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

Batch

In [ ]:
responses = llm.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

## Tool calling

### Basics

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

In [ ]:
model_with_tools = llm.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Loop

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)